# Backtester v1 — interactive checks

Run all cells with **Python (HW4)**. These small synthetic examples have known answers and need no database. They check the engine, not whether a researched strategy performs well.

You can change budgets, fees, prices, and dates below. Each scenario creates a fresh account and strategy, so it can be rerun. After changing the Python modules, restart the kernel and run all cells again.

In [ ]:
from pathlib import Path
import sys
import unittest

# Locate the submission whether VS Code starts here or at the repository root.
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
candidates += [p / 'David_MartinsHuber' for p in [cwd, *cwd.parents]]
candidates += [p / 'Homeworks/HW4/David_MartinsHuber' for p in [cwd, *cwd.parents]]
submission = next(p for p in candidates if (p / 'backtesting/engine.py').is_file())
if str(submission) not in sys.path:
    sys.path.insert(0, str(submission))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from backtesting import Backtester, OMS, Phase, Strategy
from tests.test_backtesting import BuyOnce, inputs, BacktestingChecks

check = unittest.TestCase()
print(f'Using backtester from {submission / "backtesting"}')

## 1. Order acceptance and fill booking

Expected: reserving $100 leaves actual cash at $100 and available cash at zero. A fill at $20 with no fees purchases five units. A mismatched quote and a duplicate fill must both be rejected. The strategy ID should survive into the fill and lot.

In [ ]:
oms = OMS(initial_cash=100, fee_bps=0)
order_id = oms.submit_buy('notebook_strategy', 'A', 100, '2020-01-03', '2020-01-06')
assert oms.cash == 100 and oms.available_cash == 0
print('Reserved:', {'cash': oms.cash, 'available_cash': oms.available_cash})

with check.assertRaises(ValueError):
    oms.fill(order_id, symbol='A', quote_date='2020-01-03', price=20, session='2020-01-06')

oms.fill(order_id, symbol='A', quote_date='2020-01-06', price=20, session='2020-01-06')
assert oms.lots[order_id].units == 5
assert oms.lots[order_id].strategy_id == 'notebook_strategy'
with check.assertRaises(ValueError):
    oms.fill(order_id, symbol='A', quote_date='2020-01-06', price=20, session='2020-01-06')
display(pd.DataFrame(oms.order_records()))
display(pd.DataFrame(oms.fills))

## 2. Flat-price round trip with fees

Buy with a $100 inclusive budget at $10; sell after 90 calendar days. With 1% fees per side, expected final cash is `100 × 0.99 / 1.01`, about $98.02. The loss must equal fees. This fictional calendar uses weekdays; real runs use the supplied benchmark market calendar.

In [ ]:
dates, prices, benchmark, membership = inputs()
engine = Backtester(prices, benchmark, membership, BuyOnce(), initial_cash=100, fee_bps=100)
result = engine.run(dates[1], dates[-1])
expected_cash = 100 * 0.99 / 1.01
assert np.isclose(result.history.cash.iloc[-1], expected_cash)
assert np.isclose(result.pnl.total, -result.fills.fee.sum())
assert set(result.fills.strategy_id) == {'test_strategy'}
assert result.fills.iloc[1].trigger == 'maturity'
np.testing.assert_allclose(result.history.nav, result.history.cash + result.history.holdings_value)
flows = result.fills.groupby('date').cash_change.sum().reindex(result.history.index, fill_value=0)
np.testing.assert_allclose(result.history.cash, 100 + flows.cumsum(), atol=1e-10)
assert np.isclose(result.pnl.total, result.pnl.realized + result.pnl.unrealized)
display(result.fills)
display(result.pnl.rename('PnL'))
result.history[['nav', 'benchmark_nav']].plot(title='Flat price: fees reduce portfolio value')
plt.ylabel('Portfolio value ($)')
plt.show()

## 3. Next market close, not next calendar day

Remove Monday from the calendar. Friday's signal must fill on Tuesday. Changing Tuesday's execution price must leave the purchase budget unchanged and change only its quantity. This tests the order/execution boundary; it does not prove that every possible research model is free of look-ahead.

In [ ]:
dates, prices, benchmark, membership = inputs(6)
holiday = pd.Timestamp('2020-01-06')
prices = prices.loc[prices.date.ne(holiday)].copy()
benchmark = benchmark.loc[benchmark.date.ne(holiday)].copy()
entry = pd.Timestamp('2020-01-07')
first = Backtester(prices, benchmark, membership, BuyOnce(), fee_bps=0).run(entry, dates[-1])
changed = prices.copy()
changed.loc[changed.date.eq(entry), 'adj_close'] = 20
second = Backtester(changed, benchmark, membership, BuyOnce(), fee_bps=0).run(entry, dates[-1])
assert first.orders.budget.iloc[0] == second.orders.budget.iloc[0]
assert second.fills.signal_date.iloc[0] == pd.Timestamp('2020-01-03')
assert second.fills.date.iloc[0] == entry
assert first.fills.units.iloc[0] == 2 * second.fills.units.iloc[0]
display(pd.DataFrame({'price': [first.fills.price.iloc[0], second.fills.price.iloc[0]],
                      'units': [first.fills.units.iloc[0], second.fills.units.iloc[0]],
                      'budget': [first.orders.budget.iloc[0], second.orders.budget.iloc[0]]},
                     index=['original quote', 'changed quote']))

## 4. Inspect a refit and its historical cutoff

The second training row has an unfinished label and must be excluded. This inspection strategy stores the supplied data rather than fitting a real model. It makes the boundary visible: one row reaches refitting, and no orders or cash movements occur.

In [ ]:
class InspectRefit(Strategy):
    def needs_refit(self, context):
        return context.model_version == 0

    def refit_settings(self):
        return {'purpose': 'inspect cutoff', 'window': 'expanding'}

    def refit(self, data, context):
        self.received = data
        self.latest_price_date = context.prices.index.max()

    def propose_buys(self, context):
        return {}

dates, prices, benchmark, membership = inputs(5)
panel = pd.DataFrame({
    'date': pd.to_datetime(['2019-09-01', '2019-10-01']),
    'symbol': ['A', 'A'],
    'available_date': pd.to_datetime(['2019-09-01', '2019-10-01']),
    'label_end': [dates[0] - pd.Timedelta(days=1), dates[-1]],
    'target': [0.1, 999.0],
    'feature': [1.0, 999.0],
})
refit_engine = Backtester(prices, benchmark, membership, InspectRefit('refit_demo'),
                         training_panel=panel, feature_columns=['feature'])
refit_result = refit_engine.run(dates[1], dates[-1])
assert refit_engine.strategy.received.y.tolist() == [0.1]
assert refit_engine.strategy.latest_price_date == dates[0]
assert refit_result.fills.empty
assert refit_result.history.cash.eq(1_000_000).all()
assert refit_engine.phase == Phase.COMPLETE
display(refit_engine.strategy.received.audit)
display(refit_engine.strategy.received.X)
display(refit_result.refits)
display(refit_result.phases)

## 5. Synchronized shadow portfolios

Two strategies have independent accounts and generate their own walk-forward returns. The optimizer sees completed returns only. This synthetic example warms up the shadows before starting the main account; it is an interface/accounting demonstration, not a researched trading result. Weight changes affect new purchases and never close existing lots early.

In [ ]:
from backtesting import ShadowPortfolioBacktester, MinimumVarianceOptimizer, BacktestAnalytics

class NotebookPulseStrategy(Strategy):
    def __init__(self, strategy_id, symbol, every):
        super().__init__(strategy_id)
        self.symbol, self.every, self.decisions = symbol, every, 0

    def propose_buys(self, context):
        self.decisions += 1
        if self.decisions % self.every != 1:
            return {}
        budget = context.portfolio.available_cash / 4
        return {self.symbol: budget} if budget > 1e-8 else {}

shadow_dates, base_prices, shadow_benchmark, base_membership = inputs(80)
x = np.arange(len(shadow_dates))
a_prices = base_prices.copy()
a_prices['adj_close'] = 10 * np.exp(.001 * x + .03 * np.sin(x / 4))
b_prices = base_prices.assign(symbol='B', adj_close=12 * np.exp(.0005 * x + .02 * np.cos(x / 7)))
shadow_prices = pd.concat([a_prices, b_prices], ignore_index=True)
shadow_membership = pd.concat([base_membership, base_membership.assign(symbol='B')], ignore_index=True)
shadow_benchmark['adj_close'] = 100 * np.exp(.0003 * x + .005 * np.sin(x / 8))

coordinator = ShadowPortfolioBacktester(
    shadow_prices, shadow_benchmark, shadow_membership,
    shadows=[NotebookPulseStrategy('pulse_A', 'A', 5), NotebookPulseStrategy('pulse_B', 'B', 10)],
    optimizer=MinimumVarianceOptimizer(lookback=20, min_observations=10),
)
bundle = coordinator.run(shadow_dates[16], shadow_dates[-1], shadow_start=shadow_dates[1])
assert bundle.allocations.return_cutoff.le(bundle.allocations.signal_date).all()
assert bundle.portfolio.history.cash.ge(0).all()
np.testing.assert_allclose(bundle.portfolio.history.nav,
                           bundle.portfolio.history.cash + bundle.portfolio.history.holdings_value)
display(BacktestAnalytics(bundle.portfolio).summary())
display(bundle.allocations.head())
display(bundle.contributions.head())
weights = bundle.allocations.pivot(index='signal_date', columns='component_id', values='weight')
weights.plot(title='Strategy weights based on completed shadow history', ylabel='Weight')
plt.show()
BacktestAnalytics(bundle.portfolio).plot_cumulative_returns()
plt.show()

## 6. Risk review before OMS submission

This synthetic strategy tries to buy repeatedly. A 25% signal-close name cap limits additions; a later price drop triggers a drawdown entry gate. Volatility scaling uses completed main-account returns. Existing lots still mature after 90 calendar days: the gate never liquidates them early. Observed weights in this particular scenario stay below the cap because prices only fall; the class does not guarantee that after arbitrary market moves or fees.

In [ ]:
from backtesting import RiskManager, DrawdownGate, VolatilityTarget, SingleNameCap

risk_dates, risk_prices, risk_benchmark, risk_membership = inputs(80)
risk_prices['adj_close'] = np.where(np.arange(len(risk_dates)) >= 10, 5., 10.)
controls = RiskManager([
    SingleNameCap(.25),  # Policies are sorted by priority, not this input order.
    VolatilityTarget(.15, lookback=10, min_observations=5, warmup='allow'),
    DrawdownGate(.10, resume_drawdown=.04),
])
risk_engine = Backtester(risk_prices, risk_benchmark, risk_membership,
                        NotebookPulseStrategy('risk_demo', 'A', 2),
                        initial_cash=1000, fee_bps=0, risk_manager=controls)
risk_result = risk_engine.run(risk_dates[1], risk_dates[-1])
assert risk_result.risk.budget_after.le(risk_result.risk.budget_before).all()
assert risk_result.history.cash.ge(0).all()
sales = risk_result.fills.loc[risk_result.fills.side.eq('SELL')]
assert sales.date.ge(sales.due_date).all()
assert risk_result.risk.reason.eq('drawdown_restriction').any()
display(risk_result.fills)
display(risk_result.risk[['signal_date', 'policy', 'symbol', 'budget_before', 'budget_after', 'reason']].head(15))
display(risk_result.risk.loc[risk_result.risk.reason.eq('drawdown_restriction')].head())
risk_result.history[['nav', 'cash', 'holdings_value']].plot(title='Entry restrictions preserve mandatory holding periods')
plt.ylabel('Account value ($)')
plt.show()

## 7. Run the complete regression suite

This also checks missing entry prices, bounded stale marks, independent lots, delayed maturity sales, failed refits, universe enforcement, and repeated-run rejection. The suite is loaded afresh so you can rerun this cell.

This includes loader quarantine and portfolio analysis checks: known alpha/beta, historical VaR quantiles, drawdown, Sharpe, and benchmark comparison.

The suite also checks synchronized shadows, warm-up, historical refits, immutable return prefixes, allocation constraints, and independence from future prices and labels.

Risk checks cover fee-inclusive cap room, drawdown state, volatility scaling, chronological inputs, cash limits, no early sales, and component attribution after approvals.

In [ ]:
suite = unittest.defaultTestLoader.discover(str(submission / 'tests'))
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), 'Review the failing cases above before using portfolio results.' 

## Further experiments

Add new scenarios in cells below. State the expected cash, quantity, fill date, or error before running. Use a new engine for every comparison. Synthetic checks support the accounting and timing implementation; real database quality and research feature availability still require separate inspection.

In [ ]:
# Your next hand-verifiable scenario goes here.